In [5]:
import pandas as pd
import bioframe as bf

In [12]:
REQUIRED_LENGTH = 1280

In [4]:
epdnew = pd.read_table('human_epdnew_fH7d2.bed', 
                       header=None, 
                       names=["chrom", "start", "end", "name", "length", "strand"])
genes = pd.read_table('genes.bed')
genes = genes[genes.type == '"protein_coding"']
genes.head()

,chrom,start,end,strand,type
10,chr1,65418,71585,+,"""protein_coding"""
30,chr1,450739,451678,-,"""protein_coding"""
46,chr1,685715,686654,-,"""protein_coding"""
68,chr1,923922,944575,+,"""protein_coding"""
69,chr1,944202,959309,-,"""protein_coding"""


In [6]:
right = bf.closest(epdnew, genes, ignore_upstream=True, suffixes=('', "_right"))
right.rename({'distance': 'distance_right'}, axis=1,  inplace=True)

In [7]:
left = bf.closest(right, genes, ignore_downstream=True, suffixes=('', '_left'))
left.rename({'distance': 'distance_left'}, axis=1,  inplace=True)

In [8]:
total = left

In [10]:
mask = (total['distance_left'] != 0) | (total['distance_right'] != 0)
mask.sum()


3345

In [11]:
mask = total['distance_left'] + total['distance_right'] > 1280
mask.sum()

3100

In [20]:
total = total[mask]

In [21]:
prom_regions = []

for _, row in total.iterrows():
    if row.strand == "+":
        start = row.start
        end = row.end
        end += min(row.distance_right, REQUIRED_LENGTH - 1)        
        rest = REQUIRED_LENGTH - (end - start)
        start = start - rest
    elif row.strand == '-':
        start = row.start
        end = row.end
        start -= min(row.distance_left, REQUIRED_LENGTH - 1)
        rest = REQUIRED_LENGTH - (end-start)
        end += rest    
    else:
        raise Exception()
    prom_regions.append({'chrom': row.chrom, 'start': start, 'end': end, 'strand': row.strand})
    

In [22]:
prom_regions = pd.DataFrame(prom_regions)

In [23]:
(prom_regions['end'] - prom_regions['start']).value_counts()

1280    3100
Name: count, dtype: int64

In [24]:
prom_regions

,chrom,start,end,strand
0,chr1,1206592,1207872,-
1,chr1,1239638,1240918,-
2,chr1,1375207,1376487,-
3,chr1,1692795,1694075,-
4,chr1,1726437,1727717,-
...,...,...,...,...
3095,chrX,153944687,153945967,-
3096,chrX,154019902,154021182,-
3097,chrX,154619282,154620562,-
3098,chrX,154653579,154654859,-


In [28]:
prom_regions.to_csv("promoters.bed", sep="\t")

In [29]:
!head promoters.bed

	chrom	start	end	strand
0	chr1	1206592	1207872	-
1	chr1	1239638	1240918	-
2	chr1	1375207	1376487	-
3	chr1	1692795	1694075	-
4	chr1	1726437	1727717	-
5	chr1	1744502	1745782	-
6	chr1	1744722	1746002	-
7	chr1	1744997	1746277	-
8	chr1	1780457	1781737	-
